# NIS Demo: A Free-of-Charge Trend-Signal Credibility Filter

**What this notebook shows:** a normalized, chi-square-distributed Kalman-filter innovation (NIS) —
a 50-year-old engineering fault-detection statistic — used as a credibility filter for an
established trend-following signal (SMA200×EMA50 crossover). No custom architecture, no proprietary
data: a plain-vanilla Kalman filter (constant measurement noise, no volume-awareness) and 15 public,
liquid US stocks downloaded from Yahoo Finance.

**Relationship to the paper:** the paper's reported figures are computed on a larger, licensed EODHD
panel (101 curated tickers, and a robustness check on ~1,100 tickers) that cannot be redistributed
here for licensing reasons. This notebook reproduces the *method* end-to-end on a small, freely
redistributable panel so anyone can verify the mechanism and adapt it to their own data.

**Core claim tested here:** among days where a trend-following crossover just fired, days with an
unusually high NIS ("hot") are substantially more often genuine trend reversals than days with a
low NIS ("cold") — a free-of-charge quality filter on a signal you may already be computing.


## 1. Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
from nis_core import (directional_change, sma_ema_crossings, kalman_plain_vanilla,
                       cusum_from_std_innovations, naive_heuristic, amihud_illiquidity,
                       split_into_continuous_segments, logistic_regression_manual,
                       threshold_sweep)

DATA_PATH = REPO_ROOT / "data" / "demo_panel.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Run `python scripts/download_demo_data.py` first."
    )

df_raw = pd.read_parquet(DATA_PATH)
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f"Loaded {len(df_raw)} rows, {df_raw['ticker'].nunique()} tickers, "
      f"{df_raw['date'].min().date()} to {df_raw['date'].max().date()}")

## 2. Parameters (identical to the paper's generic setup)

In [ ]:
DC_DELTA = 0.10          # Directional-Change ground truth threshold (10%)
MATCH_WINDOW = 10        # +/- days for counting a crossover as a "hit"
CAUSAL_WINDOW = 15       # backward-only window for NIS/CUSUM (no look-ahead)
CUSUM_WINDOW = 20        # rolling window for the CUSUM statistic
Q_PARAM = 1e-4           # Kalman filter process noise
R_CONST = 1e-4           # Kalman filter measurement noise (CONSTANT -- plain vanilla, no volume-awareness)
MAX_GAP_DAYS = 10        # segment break threshold (protects against seam artifacts)

print(f"DC_DELTA={DC_DELTA}, MATCH_WINDOW={MATCH_WINDOW}, CAUSAL_WINDOW={CAUSAL_WINDOW}, "
      f"CUSUM_WINDOW={CUSUM_WINDOW}")

## 3. Per-ticker pipeline (segmented, causal, VA-free)

In [ ]:
def process_ticker(sub):
    sub = sub.reset_index(drop=True)
    prices_all = sub['close'].values.astype(float)
    volume_all = sub['volume'].values.astype(float)
    dates_all = sub['date'].values
    ticker_name = sub['ticker'].values[0]
    if len(prices_all) < 260 or np.any(np.isnan(prices_all)):
        return None

    segments = split_into_continuous_segments(dates_all, max_gap_days=MAX_GAP_DAYS)
    rows = []
    for seg_start, seg_end in segments:
        prices = prices_all[seg_start:seg_end]
        volume = volume_all[seg_start:seg_end]
        dates = dates_all[seg_start:seg_end]
        if len(prices) < 260:
            continue
        log_prices = np.log(prices)

        dc_events = directional_change(prices, delta=DC_DELTA)
        dc_extreme_idx = np.array([e[1] for e in dc_events])
        cross_idx = sma_ema_crossings(prices)
        if len(dc_extreme_idx) == 0 or len(cross_idx) == 0:
            continue

        nis, std_innov = kalman_plain_vanilla(log_prices, Q=Q_PARAM, R=R_CONST)
        cusum = cusum_from_std_innovations(std_innov, window=CUSUM_WINDOW)
        heur = naive_heuristic(log_prices, volume, window=20)
        amihud = amihud_illiquidity(log_prices, volume, window=20)

        for ci in cross_idx:
            lo = ci - CAUSAL_WINDOW + 1
            if lo < 0:
                continue
            hit = bool(np.any(np.abs(dc_extreme_idx - ci) <= MATCH_WINDOW))
            w_nis = nis[lo:ci+1]; w_cusum = cusum[lo:ci+1]
            w_heur = heur[lo:ci+1]; w_amihud = amihud[lo:ci+1]
            if (np.any(np.isnan(w_nis)) or np.any(np.isnan(w_cusum)) or
                    np.any(np.isnan(w_heur)) or np.any(np.isnan(w_amihud))):
                continue
            pre_trend = np.log(prices[ci]) - np.log(prices[ci-20]) if ci >= 20 else 0.0
            rows.append(dict(ticker=ticker_name, signal_date=dates[ci], is_hit=hit,
                              max_nis=np.max(w_nis), max_cusum=np.max(w_cusum),
                              max_heur=np.max(w_heur), max_amihud=np.max(w_amihud),
                              pre_trend_20d=pre_trend))
    return rows if rows else None

print("Pipeline defined.")

## 4. Run over the demo panel

In [ ]:
t0 = time.time()
universe = df_raw['ticker'].unique().tolist()
all_rows = []
for tk in universe:
    sub = df_raw[df_raw['ticker'] == tk]
    rows = process_ticker(sub)
    if rows:
        all_rows.extend(rows)

df_signals = pd.DataFrame(all_rows)
print(f"Done in {time.time()-t0:.1f}s. {len(df_signals)} signals across {df_signals['ticker'].nunique()} tickers.")
print(f"Base hit rate (no filter): {100*df_signals['is_hit'].mean():.1f}%")

for col in ['max_nis', 'max_cusum', 'max_heur', 'max_amihud']:
    thr = np.percentile(df_signals[col], 95)
    df_signals[col.replace('max_', '') + '_hot'] = df_signals[col] > thr

## 5. Does NIS discriminate hits from false alarms?

In [ ]:
y = df_signals['is_hit'].astype(float).values
pre_trend = df_signals['pre_trend_20d'].fillna(0).values

print("--- Individual logistic regressions: is_hit ~ hot + pre_trend_20d ---")
for name, col in [('NIS', 'nis_hot'), ('CUSUM', 'cusum_hot'),
                   ('Heuristic', 'heur_hot'), ('Amihud', 'amihud_hot')]:
    X = np.column_stack([np.ones(len(df_signals)), df_signals[col].astype(float).values, pre_trend])
    beta, pvals = logistic_regression_manual(X, y)
    sig = '***' if pvals[1] < 0.001 else ('**' if pvals[1] < 0.01 else ('*' if pvals[1] < 0.05 else ''))
    print(f"  {name:10s}  OR={np.exp(beta[1]):.3f}  p={pvals[1]:.4f} {sig}")

hit_rate_hot = df_signals.loc[df_signals.nis_hot, 'is_hit'].mean()
hit_rate_cold = df_signals.loc[~df_signals.nis_hot, 'is_hit'].mean()
print(f"\nHit rate when NIS is 'hot': {100*hit_rate_hot:.1f}%")
print(f"Hit rate when NIS is 'cold': {100*hit_rate_cold:.1f}%")

## 6. Threshold sweep: the practical false-alarm/hit trade-off

In [ ]:
sweep_df, fa_before = threshold_sweep(df_signals, 'max_nis', 'is_hit')
print(f"Original false-alarm rate (no filter): {fa_before:.1f}%\n")
print(sweep_df.to_string(index=False))

## 7. A few concrete examples

In [ ]:
print("--- 5 false alarms the causal NIS filter would have flagged as 'cold' (untrustworthy) ---")
fa_examples = df_signals[~df_signals['is_hit']].sort_values('max_nis').head(5)
for _, r in fa_examples.iterrows():
    print(f"  {r['ticker']:6s}  {pd.Timestamp(r['signal_date']).date()}  max_nis={r['max_nis']:.4f}")

print("\n--- Honestly: a real hit the filter would have wrongly discarded ---")
missed = df_signals[df_signals['is_hit'] & ~df_signals['nis_hot']].sort_values('max_nis').head(3)
for _, r in missed.iterrows():
    print(f"  {r['ticker']:6s}  {pd.Timestamp(r['signal_date']).date()}  max_nis={r['max_nis']:.4f}")

## References

- Mehra, R.K. & Peschon, J. (1971). *An Innovations Approach to Fault Detection and Diagnosis in
  Dynamic Systems.* Automatica, 7(5), 637–640.
- Bar-Shalom, Y., Li, X.R. & Kirubarajan, T. (2001). *Estimation with Applications to Tracking and
  Navigation.* Wiley.
- Lee, J. et al. (2023). *Navigation safety assurance of a KF-based GNSS/IMU system.* NAVIGATION, 70(4).
- Nam, G. et al. (2026). *Kalman filter innovation-based optimal integrity monitoring...* NAVIGATION, 73.

Data: Yahoo Finance, via the `yfinance` package (`scripts/download_demo_data.py`).